<a href="https://colab.research.google.com/github/DaniilDonskoy/building_maintenance_agents/blob/feature%2Fincident-data-analysis/research/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B7%D0%B0%D1%8F%D0%B2%D0%BE%D0%BA_%D0%BC%D0%B0%D1%80%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

root = Path.cwd()
if not (root / "incident_requests").exists() and (root.parent / "incident_requests").exists():
    root = root.parent

sys.path.append(str(root))

from incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)

In [2]:
sheet_id = "1ya7OgCwUC3a_BylVxideUUAe5I18Pbr4"
sheet_name = "Sheet1"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)

print(df.columns)
display(df.head(1))

Index(['Журнал заявок\nООО "Эксплуатация Главстрой-СПб"\nОтчёт сформирован 16.03.2026 09:17 Дата',
       'Время', 'Источник', 'Адрес', 'Пом.', 'Категория', 'Подкатегория',
       'Описание', 'Комментарий к выполненным работам', 'Статус заявки',
       'Желаемое время выполнения', 'Дата исполнения', 'Исполнители',
       'Координаторы', 'Перечень материалов', 'Услуги', 'Стоимость',
       'Вложения'],
      dtype='str')


,"Журнал заявок\nООО ""Эксплуатация Главстрой-СПб""\nОтчёт сформирован 16.03.2026 09:17 Дата",Время,Источник,Адрес,Пом.,Категория,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения
0,15.03.2026,23:19,Диспетчер,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Аварийная,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет


# Предпросмотр данных

In [3]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 18 columns):
 #   Column                                                                                  Non-Null Count  Dtype  
---  ------                                                                                  --------------  -----  
 0   Журнал заявок
ООО "Эксплуатация Главстрой-СПб"
Отчёт сформирован 16.03.2026 09:17 Дата  458 non-null    str    
 1   Время                                                                                   458 non-null    str    
 2   Источник                                                                                458 non-null    str    
 3   Адрес                                                                                   458 non-null    str    
 4   Пом.                                                                                    176 non-null    str    
 5   Категория                                                                         

In [4]:
display(df['Пом.'].value_counts())
display(df['Пом.'].unique()) #Пом. - это квартира? и что означают буквы в номере

Пом.
267     3
450     3
141     3
919     2
26      2
       ..
1064    1
680     1
708     1
739     1
611     1
Name: count, Length: 157, dtype: int64

<StringArray>
[  '309',   '267',   '575',     nan,    '21',     '1',   '742',   '555',
    '90',   '168',
 ...
    '9Н', '2129П',   '65Н',    '36',  '1054',  '1064',   '680',   '708',
   '739',   '611']
Length: 158, dtype: str

In [5]:
df['Источник'].value_counts() # житель = из обращения?

Источник
Диспетчер       415
Житель           35
Из обращения      8
Name: count, dtype: int64

In [6]:
df['Категория'].value_counts() # бесполезный признак

Категория
Аварийная    458
Name: count, dtype: int64

In [7]:
df['Подкатегория'].value_counts()

Подкатегория
Лифт                214
Протечка            193
Электроснабжение     49
Прочие работы         2
Name: count, dtype: int64

In [8]:
df[df['Подкатегория'] == 'Прочие работы']

# ТЕЧЬ РАДИАТОРА В КОМНАТЕ попала в 'Прочие работы' -> тип заявки лучше узнавать из описания

,"Журнал заявок\nООО ""Эксплуатация Главстрой-СПб""\nОтчёт сформирован 16.03.2026 09:17 Дата",Время,Источник,Адрес,Пом.,Категория,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения
195,09.03.2026,11:18,Диспетчер,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 1 лит. А",64Н,Аварийная,Прочие работы,"КП, СТУДИЯ КРАСОТЫ ""ЛАЗЕР ПРО ЛАБ""- ВЫТЕКАНИЕ ИЗ УНИТАЗА 89811614344\nГИЛЬДИЯ6Промыли лежак канализации в подвале кипятком 12 метров. требуется убрать воду в подвале с 5 парадной по 4ю",NaN,Принята к исполнению,с 10:00 10.03.2026 по 12:00 10.03.2026,NaN,NaN,ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА,NaN,NaN,0,Нет
298,05.03.2026,17:37,Диспетчер,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 2 лит. А",1143,Аварийная,Прочие работы,ТЕЧЬ РАДИАТОРА В КОМНАТЕ 89312870024\n20:26 Пребрали соединение батареи радиатора,NaN,Принята к исполнению,с 17:45 05.03.2026 по 18:00 05.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Главный инженер Софронов Андрей Иванович",NaN,NaN,0,Нет


In [9]:
df['Комментарий к выполненным работам'].isna().sum()

np.int64(458)

In [10]:
df['Статус заявки'].value_counts()

Статус заявки
Принята к исполнению    458
Name: count, dtype: int64

In [11]:
df['Дата исполнения'].isna().sum() # мб полезный признак, узнаем время на работу

np.int64(458)

In [12]:
df['Исполнители'].isna().sum() # полезно, чтобы узнать количество задействованных людей

np.int64(458)

In [13]:
df['Координаторы'].value_counts() #заполнено, но непонятно как использовать

Координаторы
Спецтрест 27 – ЛИФТ Шкапцов А Л, Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович                                                           91
ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА                                                                                                                                                                                                                     84
Ведущий инженер по подъемно-транспортному оборудованию Гореликов Павел Вячеславович, Инженер по подъемно-транспортному оборудованию Горюнов Сергей Александрович, Спецтрест 27 – ЛИФТ Шкапцов А Л                                                           62
ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Главный инженер Софронов Андрей Иванович                                                                                                                                             

In [14]:
df['Перечень материалов'].isna().sum()

np.int64(458)

In [15]:
df['Услуги'].isna().sum()

np.int64(458)

In [16]:
df['Стоимость'].value_counts()

Стоимость
0    458
Name: count, dtype: int64

In [17]:
df['Вложения'].value_counts() # Что здесь имеется в виду? что означает да/нет?

Вложения
Нет    434
Да      24
Name: count, dtype: int64

# Базовые статистики по гвс + хвс

In [12]:
processor = IncidentRequestsPreprocessor(df)

incident_patterns = {
    "Замена стояка ГВС": r"(стояк).*(гвс|горяч)",
    "Замена трубы ГВС": r"(труб|трубопровод).*(гвс|горяч)",

    "Замена стояка ХВС": r"(стояк).*(хвс|холодн)",
    "Замена трубы ХВС": r"(труб|трубопровод).*(хвс|холодн)",
}

cols2drop = ['Комментарий к выполненным работам', 'Статус заявки', 'Координаторы']

processed_df = processor.preprocess(cols2drop=cols2drop, incident_patterns=incident_patterns)
print(len(processed_df))
display(processed_df.head(1))

458


,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения,Тип инцидента
0,15.03.2026,23:19,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет,Замена стояка ГВС


In [ ]:
processed_df = processed_df[processed_df['Подкатегория'] == 'Протечка']
print(len(processed_df))

193


In [17]:
df = processed_df.copy()

# Ensure datetime columns
df['Дата'] = pd.to_datetime(df['Дата'], format='%d.%m.%Y')
df['Час'] = pd.to_datetime(df['Время'], format='%H:%M').dt.hour

# Day of week
df['День недели'] = df['Дата'].dt.day_name()

# 1. Incidents per address
incidents_per_address = df.groupby('Адрес').size().sort_values(ascending=False).reset_index(name='Количество')
print("=== Инциденты по адресам ===")
display(incidents_per_address)

# 2. Incidents per weekday
incidents_per_weekday = df.groupby('День недели').size().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
).reset_index(name='Количество')
print("\n=== Инциденты по дням недели ===")
display(incidents_per_weekday)

# 3. Incidents per hour (0-23)
incidents_per_hour = df.groupby('Час').size().reindex(range(24), fill_value=0).reset_index(name='Количество')
print("\n=== Инциденты по часам ===")
display(incidents_per_hour)

# Detailed per address with incident types
address_incidents = df.groupby(['Адрес', 'Тип инцидента']).size().unstack(fill_value=0)
print("\n=== Детально по адресам ===")
display(address_incidents)

=== Инциденты по адресам ===


,Адрес,Количество
0,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 8 лит. А",14
1,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 2 лит. А",12
2,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 1 лит. А",10
3,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 3 стр. 1",10
4,"г Санкт-Петербург, п Парголово, ул Заречная, д. 45 корп. 2 стр. 1",9
5,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 4 лит. А",9
6,"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 5 лит. А",8
7,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 16 корп. 1 лит. А",7
8,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 12 корп. 1 лит. А",5
9,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 13 лит. А",5



=== Инциденты по дням недели ===


,День недели,Количество
0,Monday,26
1,Tuesday,28
2,Wednesday,30
3,Thursday,33
4,Friday,18
5,Saturday,17
6,Sunday,41



=== Инциденты по часам ===


,Час,Количество
0,0,4
1,1,4
2,2,3
3,3,3
4,4,0
5,5,1
6,6,3
7,7,3
8,8,3
9,9,13



=== Детально по адресам ===


Тип инцидента,Замена стояка ГВС,Замена стояка ХВС,Замена трубы ГВС,Прочее
Адрес,,,,
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 14 корп. 1 стр. 1",0,0,0,1
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 18 корп. 1 стр. 1",0,0,0,1
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 20 корп. 1 стр. 1",2,0,0,0
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 24 стр. 1",0,0,0,3
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 36 корп. 1 стр. 1",2,0,0,2
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 38 корп. 3 стр. 1",0,0,0,1
"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 11 корп. 1 стр. 1",0,0,0,5
"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 13 корп. 1 стр. 1",0,0,0,5
"г Санкт-Петербург, п Парголово, ул Валерия Гаврилина, д. 3 корп. 1 лит. А",0,0,0,4


# Парсинг инцидентов

In [8]:
processor = IncidentRequestsPreprocessor(df)

processed_df = processor.preprocess()
display(processed_df.head(2))

,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Комментарий к выполненным работам,Статус заявки,Желаемое время выполнения,Дата исполнения,Исполнители,Координаторы,Перечень материалов,Услуги,Стоимость,Вложения,Тип инцидента
0,15.03.2026,23:19,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",NaN,Принята к исполнению,с 09:00 16.03.2026 по 13:00 16.03.2026,NaN,NaN,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович",NaN,NaN,0,Нет,Утечка стояка
1,15.03.2026,21:32,"г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",267,Протечка,течь с потолка-ТЕЧЬ С КРОВЛИ,NaN,Принята к исполнению,с 09:00 16.03.2026 по 15:00 16.03.2026,NaN,NaN,Начальник отделения Григорьев Игорь Валерьевич,NaN,NaN,0,Да,Прочее


In [19]:
incidents_json = processor.getIncidentsData()
print(json.dumps(incidents_json["metadata"], ensure_ascii=False, indent=2))
print(json.dumps(incidents_json["incidents"][:3], ensure_ascii=False, indent=2))

{
  "total_incidents": 458,
  "gvs_incidents": 100,
  "hvs_incidents": 38,
  "lift_incidents": 171
}
[
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "23:19",
    "address": "г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",
    "incident_location": "309",
    "subcategory": "Протечка",
    "incident_type": "Утечка стояка",
    "performers": null,
    "materials": null,
    "cost": 0
  },
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "21:32",
    "address": "г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",
    "incident_location": "267",
    "subcategory": "Протечка",
    "incident_type": "Прочее",
    "performers": null,
    "materials": null,
    "cost": 0
  },
  {
    "date_start": "15.03.2026",
    "date_end": null,
    "incident_time_start": "20:48",
    "address": "г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 1 лит. А",
    "incident_location": "575",
    "

In [20]:
processor.getOverview()

,incident,count
0,Замена ХВС на уровне техэтажа,0
1,Замена главных стояков ГВС,0
2,Замена трубы ГВС,6
3,Замена трубы ХВС,0
4,Замена розлива,10
5,Анализ труб,2
6,Утечка радиатора,8
7,Утечка стояка,79
8,Прочее,353
9,Не определен,0
